# 🔬 Google Colab GPU Runner: CNN Baselines (B0–B4)
**Project:** Does Synthetic Data Really Help? Minority-Class Skin Lesion Benchmark  
**Target Minority:** Dermatofibroma (`df`, 115 images in HAM10000)  
**Primary Metric:** Minority-Class Recall (reported with Mean $\pm$ Std and 95% Bootstrap CIs)  

> **Instructions:** Ensure GPU is selected: `Runtime` -> `Change runtime type` -> `T4 GPU`.


In [ ]:
# 1. Verify GPU Acceleration
import torch
assert torch.cuda.is_available(), 'GPU not detected! Go to Runtime -> Change runtime type -> Select T4 GPU.'
print(f'Active GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')


In [ ]:
# 2. Mount Google Drive for Automatic Checkpoint & Results Persistence
from google.colab import drive
import os
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/rare_class_benchmark'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Google Drive persistent storage ready at: {DRIVE_DIR}')


In [ ]:
# 3. Setup Codebase & Dependencies
import os, sys
if os.path.exists('/content/rare-class-benchmark'):
    %cd /content/rare-class-benchmark
elif os.path.exists('scripts/run_all.py'):
    pass
elif os.path.exists('/content/scripts/run_all.py'):
    %cd /content
else:
    !git clone https://github.com/dhina-research/rare-class-skin-lesion-benchmark.git /content/rare-class-benchmark 2>/dev/null || true
    if os.path.exists('/content/rare-class-benchmark'):
        %cd /content/rare-class-benchmark
!pip install -q timm scikit-learn pandas matplotlib seaborn pyyaml tqdm


In [ ]:
# 4. Ingest HAM10000 Data
# Option A: Upload kaggle.json for fast direct unzipping
from google.colab import files
import shutil
if not os.path.exists('/root/.kaggle/kaggle.json'):
    print('Upload your kaggle.json (from kaggle.com -> Account -> Create New Token):')
    uploaded = files.upload()
    if 'kaggle.json' in uploaded:
        os.makedirs('/root/.kaggle', exist_ok=True)
        shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
        !chmod 600 /root/.kaggle/kaggle.json

if os.path.exists('/root/.kaggle/kaggle.json'):
    !kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 -p data/ --unzip
    !mkdir -p data/images
    !ln -sf data/HAM10000_images_part_1/* data/images/ 2>/dev/null || true
    !ln -sf data/HAM10000_images_part_2/* data/images/ 2>/dev/null || true
else:
    print('Running download_data.sh fallback...')
    !bash scripts/download_data.sh


In [ ]:
# 5. Generate Leak-Free Stratified Group Splits (Grouped strictly by lesion_id)
!python scripts/make_splits.py


In [ ]:
# 6. Execute CNN Baselines B0 through B4 Across All 15 Evaluations (5 folds x 3 seeds)
# Automatic per-epoch checkpointing; resumes if disconnected
!python scripts/run_all.py --arms B0 B1 B2 B3 B4 --seeds 42 1337 2024 --folds 0 1 2 3 4


In [ ]:
# 7. Aggregate Statistics and Export Results to Drive & Downloadable Zip
from src.stats import aggregate_benchmark_runs
df_summary, df_runs = aggregate_benchmark_runs('results/logs/runs.csv', 'results/tables')
display(df_summary[['arm_id', 'n_evals', 'test_minority_recall_formatted', 'test_balanced_accuracy_formatted', 'test_macro_f1_formatted']])

# Sync to Google Drive
!cp -r results/logs/runs.csv {DRIVE_DIR}/ 2>/dev/null || true
!cp -r results/tables/ {DRIVE_DIR}/ 2>/dev/null || true
!zip -r /content/baseline_results.zip results/
from google.colab import files
files.download('/content/baseline_results.zip')
